# 01 · Data audit & frozen splits (v1.0)
**Group 6 (Sparse / BM25 RAG) · Phase 1 · Member 1 – Data & Splits Lead**

This notebook (1) audits COCO 2017, (2) builds the three frozen splits with a fixed seed, and (3) runs the leakage audit.

| Split | Source | Images |
|---|---|---|
| `kb` (knowledge base) | COCO train2017 | 10,000 |
| `val` (tuning) | COCO train2017, disjoint from `kb` | 100 |
| `test` | COCO val2017, stratified | 1,000 |

**Outputs:** `splits/` (IDs, manifests, `split_meta.json`) and `report/` (`audit_report.json`, `leakage_audit.json`, `caption_policy_stats.json`).

**How to run:** set `DATA_ROOT` / `PROJECT_DIR` below (or as environment variables), then *Restart & Run All*. The last cell checks that the regenerated IDs are identical to the frozen release.

In [24]:
import os, json, random, hashlib, collections
from pathlib import Path
from PIL import Image, ImageOps
from tqdm import tqdm

# ---- Configuration (override with environment variables) ----
DATA_ROOT   = Path(os.environ.get("DATA_ROOT", "/kaggle/input/datasets/sabahesaraki/2017-2017"))
PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/kaggle/working"))
ANN = DATA_ROOT / "annotations_trainval2017" / "annotations"
OUT_SPLITS, OUT_REPORT = PROJECT_DIR / "splits", PROJECT_DIR / "report"
OUT_SPLITS.mkdir(parents=True, exist_ok=True); OUT_REPORT.mkdir(parents=True, exist_ok=True)

SEED = 42
N_KB, N_VAL = 10_000, 100
QUOTA = {"simple": 300, "medium": 400, "complex": 300}   # test = 1,000 images
N_CAPS = 5                                               # captions kept per image

# SHA-256 of the frozen v1.0 ID files (used for the final reproducibility check)
FROZEN_SHA256 = {
    "kb_ids.json":   "71d0184e4c7b6cfb82c6003af4f0efe3bcf3cd28f3bf2942022fb064282a0af4",
    "val_ids.json":  "0cf26ead61ba55690c52039b23eddf92cf2086c97383a6dd9ab33fc520ae9e3f",
    "test_ids.json": "cb24befe35a03ce734e5e8b1282b615eb74db3f1678ea3953c837528dd8f609f",
}

## 1. Locate the data

In [25]:
def find_img_dir(name):
    p = DATA_ROOT / name
    return p / name if (p / name).is_dir() else p

img_dir = {s: find_img_dir(s) for s in ["train2017", "val2017"]}
for s, p in img_dir.items():
    print(s, p, "->", len(os.listdir(p)), "files")   # expected: 118287 / 5000
print("annotation files:", sorted(f for f in os.listdir(ANN) if f.endswith(".json")))

train2017 /kaggle/input/datasets/sabahesaraki/2017-2017/train2017/train2017 -> 118287 files
val2017 /kaggle/input/datasets/sabahesaraki/2017-2017/val2017/val2017 -> 5000 files
annotation files: ['captions_train2017.json', 'captions_val2017.json', 'instances_train2017.json', 'instances_val2017.json', 'person_keypoints_train2017.json', 'person_keypoints_val2017.json']


## 2. Data audit
Checks per split: image/caption counts, duplicate image IDs, images missing on disk, orphan captions, empty captions, images with fewer than 5 captions, and the distribution of captions per image.

In [26]:
def audit(split):
    d = json.load(open(ANN / f"captions_{split}.json"))
    imgs, anns = d["images"], d["annotations"]
    ids = [i["id"] for i in imgs]
    caps = collections.defaultdict(list)
    for a in anns:
        caps[a["image_id"]].append(a["caption"])
    on_disk = set(os.listdir(img_dir[split]))
    return {
        "split": split,
        "num_images": len(ids),
        "num_captions": len(anns),
        "duplicate_image_ids": len(ids) - len(set(ids)),
        "images_missing_on_disk": sum(1 for i in imgs if i["file_name"] not in on_disk),
        "orphan_captions": len(set(caps) - set(ids)),
        "empty_captions": sum(1 for a in anns if not a["caption"].strip()),
        "images_with_less_than_5": sum(1 for i in ids if len(caps[i]) < 5),
        "captions_per_image_distribution": dict(collections.Counter(len(caps[i]) for i in ids)),
    }

reports = [audit("train2017"), audit("val2017")]
for r in reports:
    print(json.dumps(r, indent=2))
json.dump(reports, open(OUT_REPORT / "audit_report.json", "w"), indent=2)

{
  "split": "train2017",
  "num_images": 118287,
  "num_captions": 591753,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 117972,
    "6": 312,
    "7": 3
  }
}
{
  "split": "val2017",
  "num_images": 5000,
  "num_captions": 25014,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 4987,
    "6": 12,
    "7": 1
  }
}


## 3. Candidate pools and caption policy
Captions are sorted by annotation ID and the **first 5** are kept per image. Some COCO images have 6–7 captions; the extra ones are dropped, and the number of dropped captions is counted in section 6.

In [27]:
cap = {s: json.load(open(ANN / f"captions_{s}.json")) for s in ["train2017", "val2017"]}

id2file, id2caps, n_caps_raw = {}, {}, {}
for s in cap:
    id2file[s] = {i["id"]: i["file_name"] for i in cap[s]["images"]}
    tmp = collections.defaultdict(list)
    for a in sorted(cap[s]["annotations"], key=lambda a: a["id"]):
        tmp[a["image_id"]].append(a["caption"].strip())
    n_caps_raw[s] = {k: len(v) for k, v in tmp.items()}
    id2caps[s] = {k: v[:N_CAPS] for k, v in tmp.items() if len(v) >= N_CAPS}

train_pool = list(id2caps["train2017"].keys())
print("train pool:", len(train_pool))

train pool: 118287


## 4. Image integrity and selection helpers
An image is accepted only if it opens, decodes, and converts to RGB (EXIF orientation fixed) and its MD5 content hash has not been seen before. Rejected images are logged, never silently dropped.

In [28]:
def check_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()
            ImageOps.exif_transpose(im).convert("RGB")
        return True, ""
    except Exception as e:
        return False, str(e)

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

rng = random.Random(SEED)   # single RNG: the call order below must not change

def pick(pool, n, split, seen):
    pool = sorted(pool)
    rng.shuffle(pool)
    chosen, rejected = [], []
    for i in tqdm(pool, desc=f"{split} n={n}"):
        if len(chosen) == n:
            break
        p = img_dir[split] / id2file[split][i]
        ok, info = check_image(p)
        if not ok:
            rejected.append({"id": i, "reason": "unreadable", "detail": info})
            continue
        h = md5(p)
        if h in seen:
            rejected.append({"id": i, "reason": "duplicate_content"})
            continue
        seen[h] = i
        chosen.append(i)
    return chosen, rejected

## 5. Create the splits
- `kb` and `val` come from **one** shuffled train2017 pool of 10,100 images, so they cannot overlap.
- `test` comes from val2017, stratified by number of annotated objects: simple (≤3), medium (4–7), complex (≥8).

In [29]:
seen_train, seen_val = {}, {}

chosen, rej_train = pick(train_pool, N_KB + N_VAL, "train2017", seen_train)
kb_ids, val_ids = chosen[:N_KB], chosen[N_KB:]

inst = json.load(open(ANN / "instances_val2017.json"))
cnt = collections.Counter(a["image_id"] for a in inst["annotations"])
bins = {"simple": [], "medium": [], "complex": []}
for i, n in cnt.items():
    if i in id2caps["val2017"]:
        bins["simple" if n <= 3 else "medium" if n <= 7 else "complex"].append(i)
print("pool per stratum:", {k: len(v) for k, v in bins.items()})

test_ids, rej_test, test_bin = [], [], {}
for k, q in QUOTA.items():
    c, r = pick(bins[k], q, "val2017", seen_val)
    test_ids += c
    rej_test += r
    for i in c:
        test_bin[i] = k

print(len(kb_ids), len(val_ids), len(test_ids))
print("rejected train:", len(rej_train), "| rejected test:", len(rej_test))

train2017 n=10100:   9%|▊         | 10100/118287 [01:20<14:27, 124.71it/s]


pool per stratum: {'simple': 2062, 'medium': 1192, 'complex': 1698}


val2017 n=300:  18%|█▊        | 300/1698 [00:02<00:11, 119.93it/s]

10000 100 1000
rejected train: 0 | rejected test: 0


## 6. Save the frozen splits and the caption-policy statistics

In [30]:
def manifest(ids, split, extra=None):
    return [{"image_id": i, "file_name": id2file[split][i], "captions": id2caps[split][i],
             **({"stratum": extra[i]} if extra else {})} for i in sorted(ids)]

for name, ids in [("kb_ids", kb_ids), ("val_ids", val_ids), ("test_ids", test_ids)]:
    json.dump(sorted(ids), open(OUT_SPLITS / f"{name}.json", "w"))

json.dump(manifest(kb_ids, "train2017"),  open(OUT_SPLITS / "kb_manifest.json", "w"))
json.dump(manifest(val_ids, "train2017"), open(OUT_SPLITS / "val_manifest.json", "w"))
json.dump(manifest(test_ids, "val2017", test_bin), open(OUT_SPLITS / "test_manifest.json", "w"))

json.dump({"seed": SEED, "quota_test": QUOTA, "caption_policy": "first 5 captions by annotation id",
           "rejected_train": rej_train, "rejected_test": rej_test},
          open(OUT_SPLITS / "split_meta.json", "w"), indent=2)

# Caption policy: how many captions were dropped from the SELECTED images (kept in a separate file
# so that the frozen split_meta.json is not modified)
stats = {}
for name, ids, s in [("kb", kb_ids, "train2017"), ("val", val_ids, "train2017"), ("test", test_ids, "val2017")]:
    extra = [n_caps_raw[s][i] - N_CAPS for i in ids]
    stats[name] = {"images": len(ids),
                   "images_with_more_than_5_captions": sum(e > 0 for e in extra),
                   "captions_dropped": sum(extra),
                   "captions_kept": len(ids) * N_CAPS}
print(json.dumps(stats, indent=2))
json.dump(stats, open(OUT_REPORT / "caption_policy_stats.json", "w"), indent=2)
print("saved")

{
  "kb": {
    "images": 10000,
    "images_with_more_than_5_captions": 23,
    "captions_dropped": 24,
    "captions_kept": 50000
  },
  "val": {
    "images": 100,
    "images_with_more_than_5_captions": 0,
    "captions_dropped": 0,
    "captions_kept": 500
  },
  "test": {
    "images": 1000,
    "images_with_more_than_5_captions": 3,
    "captions_dropped": 3,
    "captions_kept": 5000
  }
}
saved


## 7. Leakage audit
1. **ID overlap** between every pair of splits.
2. **Content overlap** (MD5 of the image bytes): train2017 selections vs. test, and `val` vs. `kb`.
3. **Image integrity**: every selected image was opened and decoded in section 4.
4. **Caption separation**: test captions live only in `test_manifest.json`.

Since no test image is indexed in the knowledge base, a test image cannot be retrieved for itself.

In [31]:
kb, va, te = set(kb_ids), set(val_ids), set(test_ids)
kb_hashes  = {h for h, i in seen_train.items() if i in kb}
val_hashes = {h for h, i in seen_train.items() if i in va}

leak = {
    "sizes": {"kb": len(kb), "val": len(va), "test": len(te)},
    "kb_val_overlap": len(kb & va),
    "kb_test_id_overlap": len(kb & te),
    "val_test_id_overlap": len(va & te),
    "kb_test_content_overlap": len(set(seen_train) & set(seen_val)),
    "val_content_in_kb": len(val_hashes & kb_hashes),
    "rejected_unreadable": sum(r["reason"] == "unreadable" for r in rej_train + rej_test),
    "rejected_duplicates": sum(r["reason"] == "duplicate_content" for r in rej_train + rej_test),
}
assert leak["sizes"] == {"kb": N_KB, "val": N_VAL, "test": sum(QUOTA.values())}
assert leak["kb_val_overlap"] == leak["kb_test_id_overlap"] == leak["val_test_id_overlap"] == 0
assert leak["kb_test_content_overlap"] == leak["val_content_in_kb"] == 0
print(json.dumps(leak, indent=2))

# audit_report.json is rewritten from scratch, so re-running never duplicates entries
json.dump(reports + [{"selected_images_check": leak}], open(OUT_REPORT / "audit_report.json", "w"), indent=2)
json.dump(leak, open(OUT_REPORT / "leakage_audit.json", "w"), indent=2)

{
  "sizes": {
    "kb": 10000,
    "val": 100,
    "test": 1000
  },
  "kb_val_overlap": 0,
  "kb_test_id_overlap": 0,
  "val_test_id_overlap": 0,
  "kb_test_content_overlap": 0,
  "val_content_in_kb": 0,
  "rejected_unreadable": 0,
  "rejected_duplicates": 0
}


## 8. Reproducibility check against the frozen release (v1.0)

In [32]:
ok = True
for fname, expected in FROZEN_SHA256.items():
    got = hashlib.sha256((OUT_SPLITS / fname).read_bytes()).hexdigest()
    match = got == expected
    ok &= match
    print(f"{fname:15s} {'MATCH' if match else 'MISMATCH'}  {got[:16]}...")
print("\nRegenerated splits are identical to the frozen v1.0 release." if ok
      else "\nWARNING: splits differ from the frozen v1.0 release - do NOT overwrite the repository files.")

kb_ids.json     MATCH  71d0184e4c7b6cfb...
val_ids.json    MATCH  0cf26ead61ba5569...
test_ids.json   MATCH  cb24befe35a03ce7...

Regenerated splits are identical to the frozen v1.0 release.


# 9. Caption - level leakage check

In [33]:
import re
levels = {
    "exact": lambda s: s.strip(),
    "lowercase": lambda s: s.strip().lower(),
    "normalized": lambda s: " ".join(re.sub(r"[^\w\s]", " ", s.lower()).split()),
}

def caption_leak(ids, split, key):
    kb_index = collections.defaultdict(set)
    for i in kb_ids:
        for c in id2caps["train2017"][i]:
            kb_index[key(c)].add(i)
    hits = [(i, c) for i in ids for c in id2caps[split][i] if key(c) in kb_index]
    return {"matching_captions": len(hits),
            "images_affected": len({i for i, _ in hits}),
            "kb_images_involved": len({k for _, c in hits for k in kb_index[key(c)]})}

caption_leak_stats = {n: {"test_vs_kb": caption_leak(test_ids, "val2017", k),
                          "val_vs_kb": caption_leak(val_ids, "train2017", k)}
                      for n, k in levels.items()}
print(json.dumps(caption_leak_stats, indent=2))

{
  "exact": {
    "test_vs_kb": {
      "matching_captions": 91,
      "images_affected": 84,
      "kb_images_involved": 194
    },
    "val_vs_kb": {
      "matching_captions": 8,
      "images_affected": 7,
      "kb_images_involved": 15
    }
  },
  "lowercase": {
    "test_vs_kb": {
      "matching_captions": 98,
      "images_affected": 89,
      "kb_images_involved": 201
    },
    "val_vs_kb": {
      "matching_captions": 9,
      "images_affected": 8,
      "kb_images_involved": 16
    }
  },
  "normalized": {
    "test_vs_kb": {
      "matching_captions": 101,
      "images_affected": 91,
      "kb_images_involved": 207
    },
    "val_vs_kb": {
      "matching_captions": 10,
      "images_affected": 9,
      "kb_images_involved": 17
    }
  }
}


# 10. Caption exclusion

In [34]:
norm = levels["normalized"]
ref = collections.defaultdict(list)   # normalized caption -> test/val images that own it
for i in test_ids:
    for c in id2caps["val2017"][i]: ref[norm(c)].append(("test", i))
for i in val_ids:
    for c in id2caps["train2017"][i]: ref[norm(c)].append(("val", i))

excluded = [{"kb_image_id": i, "caption": c,
             "matched_with": [{"split": s, "image_id": j} for s, j in ref[norm(c)]]}
            for i in sorted(kb_ids) for c in id2caps["train2017"][i] if norm(c) in ref]

per_img = collections.Counter(e["kb_image_id"] for e in excluded)
summary = {"excluded_captions": len(excluded),
           "kb_images_affected": len(per_img),
           "max_removed_from_one_image": max(per_img.values()),
           "kb_images_left_with_0_captions": sum(v == 5 for v in per_img.values()),
           "kb_captions_remaining": N_KB * N_CAPS - len(excluded),
           "caption_leak_before_exclusion": caption_leak_stats}
print(json.dumps({k: v for k, v in summary.items() if k != "caption_leak_before_exclusion"}, indent=2))

json.dump(excluded, open(OUT_SPLITS / "excluded_captions.json", "w"), indent=1)
json.dump(summary, open(OUT_REPORT / "caption_leakage_stats.json", "w"), indent=2)

{
  "excluded_captions": 223,
  "kb_images_affected": 218,
  "max_removed_from_one_image": 2,
  "kb_images_left_with_0_captions": 0,
  "kb_captions_remaining": 49777
}


In [39]:
!PROJECT_DIR=/kaggle/working/test_run python /kaggle/input/datasets/mallakabdelbaki/src-test/data_audit.py && PROJECT_DIR=/kaggle/working/test_run python /kaggle/input/datasets/mallakabdelbaki/src-test/make_splits.py

{
  "split": "train2017",
  "num_images": 118287,
  "num_captions": 591753,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 117972,
    "6": 312,
    "7": 3
  }
}
{
  "split": "val2017",
  "num_images": 5000,
  "num_captions": 25014,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 4987,
    "6": 12,
    "7": 1
  }
}
train2017 n=10100:   9%|█▎              | 10100/118287 [01:20<14:25, 124.93it/s]
pool per stratum: {'simple': 2062, 'medium': 1192, 'complex': 1698}
val2017 n=300:  18%|████▏                   | 300/1698 [00:02<00:11, 120.15it/s]
10000 100 1000 | rejected: 0 0
{
  "sizes": {
    "kb": 10000,
    "val": 100,
    "test": 1000
  },
  "kb_val_overlap": 0,
  "kb_test_id_overlap": 0,
  "val_test_id_overlap": 